# Assignment Beta - Question 2

**Name:** Muhammad Sabeeh  
**Roll number:** 23K-0002  
**Course:** Agentic AI (AI4015)  
**Semester:** Fall 2026


# Assignment Beta - Question 2 - Muhammad Sabeeh | 23K-0002

This copy runs the baseline, all four methods and ranks 4/16/32 when **Run all** is selected on a CUDA GPU. All run controls are enabled. Kaggle working storage is used by default; no Google Drive mount is needed. Real training has not been performed on the CPU-only authoring machine. Read the included Q2 theory and four-page report.

The original reproducibility protocol is retained: seed-fixed 1800/200 split, the same train-side subset for method selection, no test-based tuning, real NF4 quantization, actual AdaLoRA schedules and resume state. Test-based best-row error analysis is descriptive and biased by selection. The general Glaive benchmark does not prove customer-support authorization safety.

## Section 1 - complete theory
For W0 of shape 4096 by 4096 and r=16, A has shape 16 by 4096 and B has shape 4096 by 16.
The adapter adds 16*4096 + 4096*16 = **131,072** trainable weights. W0 has 4096^2 = 16,777,216 weights; the proportion is **0.78125%** (bias excluded).

With W=W0+(alpha/r)BA, B=0 makes the initial update exactly zero, preserving the pretrained function. A is random so B has a nonzero initial gradient. If both were zero, both adapter gradients would be zero; if both were random, a nonzero random update would perturb outputs before any learning. Both random is not mathematically impossible, but can disrupt the first forward pass and gradients.

**QLoRA** reduces frozen-base memory with NF4 (a 4-bit codebook appropriate to near-normal weight distributions), double quantization of quantization constants and paged optimizers to manage transient optimizer memory pressure. Computation dequantizes blocks to the compute dtype; adapters remain trainable. The shared AdamW experiment does not claim to measure paged-optimizer benefits.

**DoRA** decomposes weights into a learned magnitude and normalized direction; the direction receives a low-rank update. This gives magnitude learning that plain LoRA lacks. Dense DoRA can merge into dense weights; an NF4 base requires dequantization/requantization or supported quantized merging, which may change memory, error and runtime.

**AdaLoRA** uses a singular-value-like parameterization, sensitivity/importance estimates and budget scheduling to prune less important components and distribute rank unevenly across matrices. Initial rank 12 and target average rank 8 are a global budget, not a promise of rank 8 in every layer.

Primary papers: [LoRA](https://arxiv.org/abs/2106.09685), [QLoRA](https://arxiv.org/abs/2305.14314), [DoRA](https://arxiv.org/abs/2402.09353), [AdaLoRA](https://arxiv.org/abs/2303.10512).


## 2. Environment setup

In Kaggle Settings / Session options, enable **Internet** and select an **NVIDIA GPU**.
Prefer T4 if available. The notebook uses one GPU, even on T4 x2.
Python **3.10-3.13** is supported. Python 3.13 uses PyTorch 2.8/CUDA 12.6 and
NumPy 2.2.6; Python 3.10-3.12 uses PyTorch 2.6/CUDA 12.4 and NumPy 2.0.2.
The remaining study package versions stay fixed and the exact runtime is logged.

Run setup before the experiment imports. It stops immediately if no GPU is
selected. It removes unused optional torchvision/torchaudio/librosa packages
that can cause import conflicts in this text-only study. If a changed library
was already loaded, setup requests one Python-session restart; otherwise it
checks imports in a fresh process and continues directly.

All seven training runs and original-model evaluation must finish. Do not mix
results across package, GPU, model, data or sequence-setting changes. Retain the
output checkpoints and download results before ending the session.


In [ ]:
"""Embedded Q2 setup: supported Python pins and early Kaggle GPU check."""
import importlib.metadata as metadata
import subprocess, sys, importlib

assert (3, 10) <= sys.version_info[:2] <= (3, 13), "Use Python 3.10-3.13."
try:
    gpu_probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
except FileNotFoundError as exc:
    raise RuntimeError("No NVIDIA GPU selected. In Kaggle Settings / Session options, set Accelerator to GPU (prefer T4) and enable Internet, then restart the session.") from exc
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError("NVIDIA GPU unavailable. Kaggle Accelerator must be GPU, not None or TPU. Enable a GPU and restart the session.")
print("Detected GPU:", gpu_probe.stdout.strip())

PY313 = sys.version_info[:2] == (3, 13)
TORCH_VERSION = "2.8.0" if PY313 else "2.6.0"
TORCH_CUDA = "cu126" if PY313 else "cu124"
PINS = {
    "torch": TORCH_VERSION, "transformers": "4.51.3", "peft": "0.17.1",
    "accelerate": "1.7.0", "bitsandbytes": "0.46.0", "datasets": "3.6.0",
    "numpy": "2.2.6" if PY313 else "2.0.2", "pandas": "2.2.3", "matplotlib": "3.10.1",
    "scikit-learn": "1.6.1", "huggingface-hub": "0.34.4",
    "tokenizers": "0.21.1", "safetensors": "0.5.3", "pyarrow": "20.0.0",
    "jsonschema": "4.23.0",
}

def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

roots = {p: p.replace("-", "_") for p in PINS}
roots["scikit-learn"] = "sklearn"
optional_roots = ["torchvision", "torchaudio", "librosa"]
loaded_before = {r for r in [*roots.values(), *optional_roots] if r in sys.modules}
changed_packages = set()
expected_torch = TORCH_VERSION + "+" + TORCH_CUDA
if installed("torch") != expected_torch:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "torch==" + TORCH_VERSION,
                           "--index-url", "https://download.pytorch.org/whl/" + TORCH_CUDA])
    changed_packages.add("torch")

optional = [p for p in optional_roots if installed(p)]
if optional:
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", *optional])
    changed_packages.update(optional)
missing_names = [p for p,v in PINS.items() if p != "torch" and installed(p) != v]
if missing_names:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           *[f"{p}=={PINS[p]}" for p in missing_names]])
    changed_packages.update(missing_names)
importlib.invalidate_caches()

restart_roots = loaded_before & {roots.get(p,p) for p in changed_packages}
if restart_roots:
    raise RuntimeError("Installation completed. Already-loaded libraries changed: " + ", ".join(sorted(restart_roots)) + ". Restart the Python session once, then rerun from this setup cell.")

# Check the installed stack in a fresh interpreter before notebook imports.
subprocess.check_call([sys.executable, "-c",
    "import torch,numpy,transformers,peft,accelerate,bitsandbytes,datasets; "
    "assert torch.cuda.is_available(), 'CUDA unavailable: check Kaggle GPU/driver'; "
    "print('Runtime import check passed:',torch.__version__,'NumPy',numpy.__version__)"])
print("Environment ready. Continue with the GPU inspection cell.")


## 3. Runtime/GPU inspection

BF16 is enabled only when CUDA reports support. On a T4 use FP16 with GradScaler;
trainable adapter tensors stay FP32 for stable optimizer updates. The frozen base retains the
prescribed storage precision; QLoRA preparation upcasts some nonquantized tensors as PEFT requires.
CUDA must be available for assignment runs. Setup and non-GPU validation can still be inspected on CPU.


In [ ]:
import os, gc, json, math, time, random, hashlib, platform, warnings, re, ast, copy
from pathlib import Path
from collections import Counter
from contextlib import nullcontext
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers, peft, datasets, accelerate, bitsandbytes as bnb
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          GenerationConfig, get_linear_schedule_with_warmup)
from peft import (LoraConfig, AdaLoraConfig, TaskType, get_peft_model,
                  prepare_model_for_kbit_training, PeftModel)
from huggingface_hub import HfApi
from datasets import load_dataset
from jsonschema import Draft7Validator
from IPython.display import display

CUDA = torch.cuda.is_available()
DTYPE = torch.bfloat16 if CUDA and torch.cuda.is_bf16_supported() else torch.float16
HARDWARE = {
    "python": sys.version, "platform": platform.platform(), "cuda_available": CUDA,
    "torch_cuda": torch.version.cuda, "compute_dtype": str(DTYPE),
    "gpu": torch.cuda.get_device_name(0) if CUDA else None,
    "vram_gib": torch.cuda.get_device_properties(0).total_memory / 2**30 if CUDA else None,
    "compute_capability": list(torch.cuda.get_device_capability(0)) if CUDA else None,
    "versions": {p: installed(p) for p in PINS},
}
print(json.dumps(HARDWARE, indent=2))
if not CUDA:
    print("CPU detected. Dataset preparation is allowed; actual experiment runs require NVIDIA CUDA.")
else:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print("pip check:", pip_check.stdout or pip_check.stderr)
print("Kaggle-wide conflicts in unrelated preinstalled packages may appear; resolve conflicts involving PINS.")


## 4. Configuration and reproducibility

Use the same output directory after restoring checkpoints into Kaggle working storage. Every run is bound to
the model/dataset commit SHA, tokenized split hashes, training settings, package versions, GPU
and dtype. A mismatch stops reuse/resume instead of mixing incomparable results.
Two epochs, LR 2e-4, AdamW, linear schedule with 3% warmup, effective batch 8 and q/v adapters
are a modest fixed study, not a tuned optimum. All 1,800 training records are visited each epoch
(225 optimizer updates/epoch). AdamW is deliberately the same for QLoRA: paged optimizers
are possible but would introduce another treatment difference.

LoRA/QLoRA/DoRA use alpha=2×rank and dropout=.05; AdaLoRA uses alpha=2×initial_rank
to maintain initial scaling≈2 under its SVD parameterization. AdaLoRA also has its intrinsic
orthogonality penalty and adaptive rank allocation. These inherent differences are logged.
No adapter merging is performed in measured comparisons: all adapter runs are measured unmerged.


In [ ]:
USE_DRIVE = False  # Kaggle uses local working storage; no Drive mount.
OUTPUT_DIR = Path("/kaggle/working/Assignment_Beta_Q2")

CFG = {
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_id": "glaiveai/glaive-function-calling-v2",
    "model_revision": "main", "dataset_revision": "main",  # Resolved to immutable SHAs once.
    "seed": 42, "n_train": 1800, "n_test": 200, "n_selection": 180,
    "epochs": 2, "micro_batch": 1, "grad_accum": 8, "lr": 2e-4,
    "weight_decay": .01, "warmup_ratio": .03, "max_grad_norm": 1.0,
    "max_seq_length": 1024, "max_prompt_tokens": 768, "max_new_tokens": 256,
    "target_modules": ["q_proj", "v_proj"], "dropout": .05, "scaling_ratio": 2,
    "gradient_checkpointing": True, "attn_implementation": "sdpa",
    "checkpoint_every": 50, "save_resume_checkpoints": True,
    "adam_betas": [.9, .999], "adalora_orth_reg": .5,
    "selection_rule": "train-side tool_pct, exact_pct, json_pct desc; params, method asc",
    "protocol_version": "beta-q2-v1",
}
assert CFG["n_train"] == 1800 and CFG["n_test"] == 200
assert CFG["n_train"] % (CFG["micro_batch"] * CFG["grad_accum"]) == 0
assert CFG["max_prompt_tokens"] + CFG["max_new_tokens"] <= CFG["max_seq_length"]
for subdir in ("dataset", "runs", "tables", "graphs"):
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if CUDA:
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    # Some kernels can be nondeterministic; warnings are preserved in notebook output.
    torch.use_deterministic_algorithms(True, warn_only=True)

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
seed_all(CFG["seed"])

def json_default(value):
    if isinstance(value, set):
        return sorted(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Unsupported JSON value: {type(value).__name__}")

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                      allow_nan=False, default=json_default)

def digest(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()

def file_digest(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False,
                               default=json_default), encoding="utf-8")
    os.replace(temp, path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def save_jsonl(path, rows):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    with temp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(canonical(row) + "\n")
    os.replace(temp, path)

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

lock_path = OUTPUT_DIR / "protocol.json"
if lock_path.exists():
    PROTOCOL = read_json(lock_path)
    assert PROTOCOL["config"] == CFG, "Configuration changed. Use a NEW OUTPUT_DIR for a new study."
else:
    api = HfApi()
    PROTOCOL = {
        "config": CFG,
        "model_sha": api.model_info(CFG["model_id"], revision=CFG["model_revision"]).sha,
        "dataset_sha": api.dataset_info(CFG["dataset_id"], revision=CFG["dataset_revision"]).sha,
    }
    save_json(lock_path, PROTOCOL)
save_json(OUTPUT_DIR / "hardware_latest.json", HARDWARE)
(OUTPUT_DIR / "requirements_runtime.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
print("Persistent output:", OUTPUT_DIR)
print("Frozen revisions:", PROTOCOL["model_sha"], PROTOCOL["dataset_sha"])


## 5. Dataset loading and preprocessing

Source: [Glaive function calling v2](https://huggingface.co/datasets/glaiveai/glaive-function-calling-v2)
(Apache-2.0, machine-generated benchmark conversations; not verified customer traffic).
Inspection of real source samples confirmed `system` tool definitions and `chat` transcripts.
The code inspects the full loaded schema and scans **all rows**, not just the first 2,000.
It counts valid first calls to each named business tool and checks whether 2,000 examples exist
within exactly that four-tool inventory. If insufficient, it explicitly uses general genuine
benchmark calls with their original schemas/names/arguments; no relabeling is performed.
An authoring-time scan of all 112,960 source rows found **zero declarations of each of the
four exact business function names**. The runtime audit repeats this check on its frozen revision.
Even the four-tool subset would not prove safe production deployment.
The manifest also records how many tools each selected prompt offers. Single-tool prompts
make tool-name selection easier; interpret tool accuracy together with argument exact match.

One example per conversation: use only the first user turn and its immediate assistant function
call. Exclude refusals, non-call first answers, multi-call/ambiguous records, invalid schemas,
undeclared arguments, schema-invalid calls, and overlength records. Later function results and
assistant answers never enter the prompt. Extract top-level JSON schemas with JSONDecoder;
parse mixed-quoted source calls using `ast.literal_eval` only as a source-format fallback
(never `eval`). Inner argument strings must be real JSON. Preserve the original schema object.
Deduplicate normalized user messages globally, preventing exact/whitespace/case duplicates
from crossing the split; semantic paraphrases and pretraining contamination remain possible.
All filtering is structural, fixed before test predictions, not based on model performance.


In [ ]:
BUSINESS_TOOLS = {"check_order_status", "initiate_refund", "track_shipment", "escalate_to_human"}
OUTPUT_INSTRUCTION = (
    "Select the appropriate function for the user's request using only the available functions below. "
    'Return exactly one JSON object with keys "name" (string) and "arguments" (object). '
    "No markdown, explanation, or extra keys. Do not execute the function.\nAvailable functions:\n"
)

def strict_loads(text):
    def reject_constant(value):
        raise ValueError("Nonstandard JSON constant: " + value)
    def no_duplicate_keys(pairs):
        d = {}
        for k, v in pairs:
            if k in d:
                raise ValueError("Duplicate JSON key: " + k)
            d[k] = v
        return d
    return json.loads(text, parse_constant=reject_constant, object_pairs_hook=no_duplicate_keys)

def extract_tools(system):
    decoder = json.JSONDecoder()
    tail = system[system.find("{"):] if "{" in system else ""
    tools = []
    while tail.strip():
        tail = tail.lstrip(" \r\n\t,")
        if not tail:
            break
        obj, end = decoder.raw_decode(tail)
        if not isinstance(obj, dict) or not isinstance(obj.get("name"), str):
            raise ValueError("Unsupported tool schema format")
        schema = obj.get("parameters")
        if not isinstance(schema, dict) or schema.get("type") != "object":
            raise ValueError("Missing object parameter schema")
        Draft7Validator.check_schema(schema)
        tools.append(obj)
        tail = tail[end:]
    if not tools or len({t["name"] for t in tools}) != len(tools):
        raise ValueError("No tools or duplicate tool names")
    return tools

def parse_source(row, index):
    tools = extract_tools(row["system"])
    chat = row["chat"]
    m = re.match(r"^\s*USER:\s*(.*?)\s*ASSISTANT:\s*(.*?)<\|endoftext\|>", chat, re.S)
    if not m:
        raise ValueError("Unrecognized first turn")
    user, answer = m.groups()
    user = user.strip()
    if not user or any(marker in user for marker in ("FUNCTION RESPONSE:", "<functioncall>", "<|im_")):
        raise ValueError("Ambiguous user boundary")
    if not answer.strip().startswith("<functioncall>"):
        raise ValueError("First answer is not a direct function call")
    payload = answer.strip()[len("<functioncall>"):].strip()
    try:
        call = strict_loads(payload)
    except (ValueError, json.JSONDecodeError):
        call = ast.literal_eval(payload)  # Source format conversion only.
    if not isinstance(call, dict) or set(call) != {"name", "arguments"}:
        raise ValueError("Not a single supported function call")
    if isinstance(call["arguments"], str):
        call["arguments"] = strict_loads(call["arguments"])
    if not isinstance(call["arguments"], dict) or not isinstance(call["name"], str):
        raise ValueError("Bad call types")
    by_name = {t["name"]: t for t in tools}
    if call["name"] not in by_name:
        raise ValueError("Ground truth names an unavailable function")
    schema = by_name[call["name"]]["parameters"]
    Draft7Validator(schema).validate(call["arguments"])
    if set(call["arguments"]) - set(schema.get("properties", {})):
        raise ValueError("Undeclared source argument")
    # Canonical JSON roundtrip also rejects NaN and unsupported Python values.
    call = strict_loads(canonical(call))
    key = digest(re.sub(r"\s+", " ", user).casefold())
    return {"source_index": index, "id": f"glaive-{index}", "user": user,
            "tools": tools, "expected": call, "user_hash": key}

def messages_for(example):
    return [{"role": "system", "content": OUTPUT_INSTRUCTION + canonical(example["tools"])},
            {"role": "user", "content": example["user"]}]

def prompt_text(example):
    return tokenizer.apply_chat_template(messages_for(example), tokenize=False, add_generation_prompt=True)

def encode_example(example):
    messages = messages_for(example)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(messages + [
        {"role": "assistant", "content": canonical(example["expected"])}],
        tokenize=False, add_generation_prompt=False)
    prefix = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    tokens = tokenizer(full, add_special_tokens=False)["input_ids"]
    if tokens[:len(prefix)] != prefix:
        raise ValueError("Chat boundary token mismatch")
    if (len(prefix) > CFG["max_prompt_tokens"] or len(tokens) > CFG["max_seq_length"]
            or len(tokens) - len(prefix) > CFG["max_new_tokens"]):
        raise ValueError("Over token budget; no truncation allowed")
    labels = [-100] * len(prefix) + tokens[len(prefix):]
    assert any(x != -100 for x in labels)
    return {"input_ids": tokens, "attention_mask": [1] * len(tokens), "labels": labels}

tokenizer = AutoTokenizer.from_pretrained(CFG["model_id"], revision=PROTOCOL["model_sha"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.save_pretrained(OUTPUT_DIR / "dataset" / "tokenizer")

def prepare_dataset():
    folder = OUTPUT_DIR / "dataset"
    if (folder / "manifest.json").exists():
        print("Reusing frozen dataset; validation below checks hashes.")
        return
    raw = load_dataset(CFG["dataset_id"], revision=PROTOCOL["dataset_sha"], split="train")
    print("Actual source features:", raw.features)
    print("Source rows:", len(raw))
    assert {"system", "chat"}.issubset(raw.column_names), "Dataset schema changed; inspect before proceeding."
    # Show schema/types only. No test examples are displayed.
    print("First record field types:", {k: type(v).__name__ for k, v in raw[0].items()})
    rejected, seen, candidates = Counter(), set(), []
    business_call_counts = Counter()
    for index, row in enumerate(raw):
        try:
            ex = parse_source(row, index)
            if ex["user_hash"] in seen:
                rejected["Duplicate normalized user message"] += 1
                continue
            encode_example(ex)  # Structural/token-budget eligibility only.
            seen.add(ex["user_hash"])
            candidates.append(ex)
            if ex["expected"]["name"] in BUSINESS_TOOLS:
                business_call_counts[ex["expected"]["name"]] += 1
        except Exception as exc:
            rejected[type(exc).__name__ + ": " + str(exc).splitlines()[0][:100]] += 1
        if (index + 1) % 10000 == 0:
            print(f"Scanned {index+1:,}/{len(raw):,}; eligible unique examples: {len(candidates):,}")
    scenario = [x for x in candidates if x["expected"]["name"] in BUSINESS_TOOLS
                and {t["name"] for t in x["tools"]}.issubset(BUSINESS_TOOLS)]
    scenario_complete = len(scenario) >= 2000 and {
        x["expected"]["name"] for x in scenario} == BUSINESS_TOOLS
    pool = scenario if scenario_complete else candidates
    scope = "four-tool customer-support subset" if scenario_complete else "general function-calling benchmark"
    limitation = ("Four-tool subset available; coverage is logged." if scenario_complete else
                  f"Only {len(scenario)} suitable unique examples within the exact four-tool inventory; "
                  "using the assignment-permitted general benchmark with original functions. "
                  "Results do not establish accuracy on the four customer-support tools.")
    print(limitation)
    assert len(pool) >= 2000, f"Only {len(pool)} suitable examples. Stop; do not synthesize labels."
    rng = np.random.default_rng(CFG["seed"])
    selected = [pool[int(i)] for i in rng.permutation(len(pool))[:2000]]
    # Split after deduplication; one source conversation contributes at most one record.
    train, test = selected[:1800], selected[1800:]
    selection_ids = [train[int(i)]["id"] for i in rng.permutation(1800)[:CFG["n_selection"]]]
    save_jsonl(folder / "train.jsonl", train)
    save_jsonl(folder / "test.jsonl", test)
    save_json(folder / "selection_ids.json", selection_ids)
    manifest = {
        "protocol": PROTOCOL, "source_features": str(raw.features), "source_rows": len(raw),
        "eligible_unique": len(candidates), "scope": scope, "limitation": limitation,
        "business_call_counts": dict(business_call_counts), "scenario_eligible": len(scenario),
        "selected_available_tool_counts": dict(Counter(str(len(x["tools"])) for x in selected)),
        "rejections": dict(rejected), "n_train": 1800, "n_test": 200,
        "train_hash": digest(train), "test_hash": digest(test),
        "selection_ids_hash": digest(selection_ids),
        "encoded_train_hash": digest([encode_example(x) for x in train]),
        "filter_policy": "first-direct-single-call; valid original schema; unique user; no truncation",
    }
    save_json(folder / "manifest.json", manifest)  # Completion marker written last.
    del raw, candidates, pool
    gc.collect()

prepare_dataset()


## 6. Dataset validation and train/test split

Training labels cover only the assistant JSON and its chat terminator. Prompt and padding labels
are `-100`. No packing and no truncation are used. The whole 200-example test split is evaluated
every time; it never enters the DataLoader, optimizer, schedule or selection scoring.
Saved JSONL records include source row index, original tool schema, user message and true call.
The manifest records rejected records, scenario coverage and tokenization fingerprints.


In [ ]:
MANIFEST = read_json(OUTPUT_DIR / "dataset" / "manifest.json")
TRAIN = read_jsonl(OUTPUT_DIR / "dataset" / "train.jsonl")
TEST = read_jsonl(OUTPUT_DIR / "dataset" / "test.jsonl")
SELECTION_IDS = read_json(OUTPUT_DIR / "dataset" / "selection_ids.json")
assert MANIFEST["protocol"] == PROTOCOL
assert len(TRAIN) == 1800 and len(TEST) == 200
assert digest(TRAIN) == MANIFEST["train_hash"] and digest(TEST) == MANIFEST["test_hash"]
assert digest(SELECTION_IDS) == MANIFEST["selection_ids_hash"]
assert not ({x["user_hash"] for x in TRAIN} & {x["user_hash"] for x in TEST})
assert not ({x["source_index"] for x in TRAIN} & {x["source_index"] for x in TEST})
assert len({x["user_hash"] for x in TRAIN + TEST}) == 2000
train_by_id = {x["id"]: x for x in TRAIN}
SELECTION = [train_by_id[i] for i in SELECTION_IDS]
assert len(SELECTION) == CFG["n_selection"] and len(set(SELECTION_IDS)) == len(SELECTION)
ENCODED_TRAIN = [encode_example(x) for x in TRAIN]
assert digest(ENCODED_TRAIN) == MANIFEST["encoded_train_hash"]
for ex in TEST:
    encode_example(ex)
sample = ENCODED_TRAIN[0]
first_label = next(i for i, v in enumerate(sample["labels"]) if v != -100)
assert all(x == -100 for x in sample["labels"][:first_label])
print("Verified: 1,800 training / 200 test; disjoint source/user IDs; 180 training-side selection records.")
print("Benchmark scope:", MANIFEST["scope"])
print(MANIFEST["limitation"])
print("Selected prompts by available tool count:", MANIFEST["selected_available_tool_counts"])
print("Training-only decoded target:", tokenizer.decode(sample["input_ids"][first_label:]))


## 7. Base model evaluation

The actual baseline execution control is immediately after the shared definitions in Section 8
so that all evaluation functions are available. Enable that control **before any fine-tuning**;
training checks that the baseline is complete. Baseline uses BF16/FP16 original weights with no
adapter or quantization, and all 200 test examples. Training-specific metrics are N/A.

**Scoring:** `json.loads` of the whole stripped response, with no code-fence removal, substring
extraction, repairs or type coercion. Nonstandard NaN/Infinity and duplicate keys are rejected.
JSON validity is syntactic: a parsed list or wrong-shaped object is still valid JSON, but fails
the expected-call schema and gets zero tool/full-match credit. A separate `call_shape_valid`
flag captures this distinction. Expected calls have exactly `name` and `arguments` keys.
Full match recursively ignores object key order, preserves list order, preserves every value
and its Python JSON type (including integer vs float and boolean vs integer); strings are not
lowercased/trimmed. Missing/extra arguments fail. Invalid outputs get zero tool and exact credit.

**Speed:** one untimed warmup on a training example, batch size 1, greedy decoding, max 256
new tokens, identical prompt order and GPU, no merge. CUDA-synchronized generate latency
includes prefill and decoding, excludes tokenization/file I/O/model loading. Report aggregate
generated tokens/sec and requests/sec, plus mean/median latency. Variable output lengths
influence speed; treat results as workload measurements, not a controlled kernel benchmark.


## 8. Shared training/evaluation utilities

The explicit PyTorch loop uses PEFT models and a common optimizer, fixed epoch permutations,
correct accumulation, gradient clipping, mixed precision and checkpointing. AdaLoRA calls
`update_and_allocate` **after successful optimizer updates, before zeroing gradients**.
Its schedule uses the actual 450 optimizer updates (10% initial, 10% final, allocation every
update); ranks are a global budget, so individual layers need not end at exactly the target.
FP16 overflow skips do not advance optimizer/rank/scheduler steps and retry the same batch.

Resume checkpoints retain full-size trainable tensors, AdamW, scheduler, scaler, all RNG states,
next optimizer step, accumulated wall time/peak memory, logs and AdaLoRA importance EMAs/rank
mask. Final adapters use PEFT's portable saved format. This avoids trying to resume training
from AdaLoRA's compressed inference adapter with incompatible optimizer tensor shapes.
Only load `.pt` checkpoints created by your own notebook (they contain Python RNG state).


In [ ]:
def require_gpu():
    if not CUDA:
        raise RuntimeError("NVIDIA CUDA GPU required. Choose a Kaggle GPU accelerator.")
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free/total: {free/2**30:.2f}/{total/2**30:.2f} GiB")
    if free < 6 * 2**30:
        warnings.warn("Less than 6 GiB free: nonquantized training may OOM. No method will be substituted.")

def cleanup():
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()

def autocast_context():
    return torch.autocast("cuda", dtype=DTYPE) if CUDA else nullcontext()

def load_base(method):
    require_gpu()
    kwargs = dict(revision=PROTOCOL["model_sha"], torch_dtype=DTYPE,
                  device_map={"": 0}, attn_implementation=CFG["attn_implementation"])
    if method == "QLoRA":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=DTYPE)
    model = AutoModelForCausalLM.from_pretrained(CFG["model_id"], **kwargs)
    assert all(p.device.type == "cuda" for p in model.parameters()), "CPU/offload is not part of this protocol."
    if method == "QLoRA":
        assert model.is_loaded_in_4bit
        qlayers = [m for m in model.modules() if isinstance(m, bnb.nn.Linear4bit)]
        assert qlayers and all(m.weight.quant_state is not None for m in qlayers)
        assert all(m.weight.quant_state.quant_type == "nf4" for m in qlayers)
    else:
        assert not getattr(model, "is_loaded_in_4bit", False)
    return model

def adapter_config(method, rank, total_steps):
    common = dict(task_type=TaskType.CAUSAL_LM, target_modules=CFG["target_modules"],
                  lora_dropout=CFG["dropout"], bias="none")
    if method == "AdaLoRA":
        initial = math.ceil(1.5 * rank)  # 4→6, 8→12, 16→24, 32→48.
        return AdaLoraConfig(**common, init_r=initial, target_r=rank,
            lora_alpha=CFG["scaling_ratio"] * initial, total_step=total_steps,
            tinit=max(1, int(.1 * total_steps)), tfinal=max(1, int(.1 * total_steps)),
            deltaT=1, beta1=.85, beta2=.85, orth_reg_weight=CFG["adalora_orth_reg"])
    assert method in ("LoRA", "QLoRA", "DoRA")
    return LoraConfig(**common, r=rank, lora_alpha=CFG["scaling_ratio"] * rank,
                      use_dora=(method == "DoRA"))

def training_model(method, rank, total_steps):
    base = load_base(method)
    if method == "QLoRA":
        base = prepare_model_for_kbit_training(base,
            use_gradient_checkpointing=CFG["gradient_checkpointing"],
            gradient_checkpointing_kwargs={"use_reentrant": False})
    base.config.use_cache = False
    if CFG["gradient_checkpointing"]:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        base.enable_input_require_grads()
    model = get_peft_model(base, adapter_config(method, rank, total_steps))
    # Stable optimizer/GradScaler: only adapter tensors are trainable and they remain FP32.
    for name, p in model.named_parameters():
        if p.requires_grad:
            assert "lora_" in name, f"Unexpected trainable base parameter: {name}"
            p.data = p.data.float()
    if method == "DoRA":
        assert model.peft_config["default"].use_dora
        assert any("lora_magnitude_vector" in n for n, _ in model.named_parameters())
    if method == "AdaLoRA":
        assert hasattr(model.base_model, "update_and_allocate")
        assert model.peft_config["default"].init_r == math.ceil(1.5 * rank)
    model.print_trainable_parameters()
    return model

def typed_equal(a, b):
    if type(a) is not type(b):
        return False
    if isinstance(a, dict):
        return a.keys() == b.keys() and all(typed_equal(a[k], b[k]) for k in a)
    if isinstance(a, list):
        return len(a) == len(b) and all(typed_equal(x, y) for x, y in zip(a, b))
    return a == b

def score_response(text, expected):
    try:
        parsed = strict_loads(text.strip())
        valid = True
    except (ValueError, TypeError, json.JSONDecodeError):
        parsed, valid = None, False
    shape = (valid and isinstance(parsed, dict) and set(parsed) == {"name", "arguments"}
             and isinstance(parsed["name"], str) and isinstance(parsed["arguments"], dict))
    tool = bool(shape and parsed["name"] == expected["name"])
    exact = bool(shape and typed_equal(parsed, expected))
    return {"json_valid": valid, "call_shape_valid": bool(shape), "tool_correct": tool,
            "exact_match": exact, "parsed": parsed}

def eval_model(model, examples, destination):
    assert len(examples) in (200, CFG["n_selection"])
    model.eval()
    model.config.use_cache = True
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    generation = GenerationConfig(do_sample=False, num_beams=1,
        max_new_tokens=CFG["max_new_tokens"], use_cache=True,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    def generate_one(ex):
        inputs = tokenizer(prompt_text(ex), add_special_tokens=False, return_tensors="pt").to("cuda")
        assert inputs["input_ids"].shape[-1] <= CFG["max_prompt_tokens"]
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.inference_mode(), autocast_context():
            out = model.generate(**inputs, generation_config=generation)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        new = out[0, inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(new, skip_special_tokens=True), int(new.numel()), elapsed

    generate_one(TRAIN[0])  # Fixed warmup, never a test example.
    rows = []
    for i, ex in enumerate(examples):
        text, n_tokens, elapsed = generate_one(ex)
        scored = score_response(text, ex["expected"])
        rows.append({"id": ex["id"], "source_index": ex["source_index"], "user": ex["user"],
                     "expected": ex["expected"], "raw_response": text, **scored,
                     "generated_tokens": n_tokens, "generation_seconds": elapsed,
                     "hit_token_limit": n_tokens >= CFG["max_new_tokens"]})
        if (i + 1) % 25 == 0:
            print(f"Evaluated {i+1}/{len(examples)}")
    n = len(rows)
    assert [x["id"] for x in rows] == [x["id"] for x in examples]
    valid = sum(x["json_valid"] for x in rows)
    tool = sum(x["tool_correct"] for x in rows)
    exact = sum(x["exact_match"] for x in rows)
    seconds = sum(x["generation_seconds"] for x in rows)
    metrics = {"n": n, "json_valid_count": valid, "json_invalid_count": n-valid,
        "json_pct": 100*valid/n, "tool_correct_count": tool, "tool_incorrect_count": n-tool,
        "tool_pct": 100*tool/n, "exact_count": exact, "exact_pct": 100*exact/n,
        "generated_tokens": sum(x["generated_tokens"] for x in rows),
        "generation_seconds": seconds,
        "tokens_per_second": sum(x["generated_tokens"] for x in rows)/seconds,
        "requests_per_second": n/seconds, "mean_latency_seconds": seconds/n,
        "median_latency_seconds": float(np.median([x["generation_seconds"] for x in rows])),
        "truncated_count": sum(x["hit_token_limit"] for x in rows)}
    save_jsonl(destination, rows)
    return metrics

def collate_examples(records, device="cuda"):
    width = max(len(x["input_ids"]) for x in records)
    return {
        "input_ids": torch.tensor([x["input_ids"] + [tokenizer.pad_token_id]*(width-len(x["input_ids"]))
                                  for x in records], dtype=torch.long, device=device),
        "attention_mask": torch.tensor([x["attention_mask"] + [0]*(width-len(x["input_ids"]))
                                       for x in records], dtype=torch.long, device=device),
        "labels": torch.tensor([x["labels"] + [-100]*(width-len(x["input_ids"]))
                               for x in records], dtype=torch.long, device=device),
    }

def run_identity(method, rank):
    return {"protocol": PROTOCOL, "train_hash": MANIFEST["train_hash"],
        "test_hash": MANIFEST["test_hash"], "encoded_train_hash": MANIFEST["encoded_train_hash"],
        "selection_ids_hash": MANIFEST["selection_ids_hash"], "method": method, "rank": rank,
        "hardware": HARDWARE}

def run_dir(method, rank=8):
    return OUTPUT_DIR / "runs" / ("Original" if method == "Original" else f"{method}_r{rank}")

def ensure_run(method, rank):
    folder = run_dir(method, rank)
    folder.mkdir(parents=True, exist_ok=True)
    identity = run_identity(method, rank)
    path = folder / "identity.json"
    if path.exists():
        assert read_json(path) == identity, f"Run identity changed for {folder}. Use a NEW OUTPUT_DIR."
    else:
        save_json(path, identity)
    return folder

def completed(method, rank=8):
    folder = run_dir(method, rank)
    path = folder / "metrics.json"
    if not path.exists():
        return False
    assert read_json(folder / "identity.json") == run_identity(method, rank), f"Identity mismatch: {folder}"
    result = read_json(path)
    assert result["status"] == "complete" and result["test"]["n"] == 200
    predictions = read_jsonl(folder / "test_predictions.jsonl")
    assert len(predictions) == 200 and [x["id"] for x in predictions] == [x["id"] for x in TEST]
    assert digest(predictions) == result["test_predictions_hash"]
    if method != "Original":
        assert (folder / "adapter" / "adapter_model.safetensors").exists(), "Saved adapter missing."
        for name, expected_hash in result["adapter_file_hashes"].items():
            assert file_digest(folder / "adapter" / name) == expected_hash, "Adapter file changed."
    return True

def run_baseline():
    if completed("Original", 0):
        print("Baseline already complete:", run_dir("Original"))
        return
    folder = ensure_run("Original", 0)
    model = None
    try:
        seed_all(CFG["seed"])
        model = load_base("Original")
        for p in model.parameters():
            p.requires_grad_(False)
        test = eval_model(model, TEST, folder / "test_predictions.jsonl")
        result = {"status": "complete", "method": "Original", "rank": None,
            "trainable_parameters": None, "trainable_pct": None,
            "peak_gpu_gib": None, "training_minutes": None, "training_loss": None,
            "test": test, "test_predictions_hash": digest(read_jsonl(folder / "test_predictions.jsonl")),
            "identity_hash": digest(run_identity("Original", 0))}
        save_json(folder / "metrics.json", result)
        print(test)
    except torch.cuda.OutOfMemoryError as exc:
        save_json(folder / "error.json", {"error": "CUDA OOM", "stage": "baseline"})
        raise RuntimeError("Baseline OOM: use a larger GPU/new consistent protocol; no quantization substituted.") from exc
    finally:
        del model
        cleanup()


In [ ]:
RUN_BASELINE = True  # Set True and execute after setup/data. Required before any training.
if RUN_BASELINE:
    run_baseline()
else:
    print("Baseline control disabled. Set RUN_BASELINE=True to evaluate the original model.")


In [ ]:
def allocator_state(model):
    if not hasattr(model.base_model, "rankallocator"):
        return None
    alloc = model.base_model.rankallocator
    return {k: {n: t.detach().cpu() for n, t in getattr(alloc, k).items()}
            for k in ("ipt", "exp_avg_ipt", "exp_avg_unc")}

def save_resume(path, model, optimizer, scheduler, scaler, next_step, logs, seconds, peak):
    state = {
        "next_step": next_step,
        "trainable": {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad},
        "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(), "allocator": allocator_state(model),
        "rank_pattern": copy.deepcopy(model.peft_config["default"].rank_pattern),
        "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all() if CUDA else [],
        "logs": logs, "training_seconds": seconds, "peak_bytes": peak,
    }
    temp = path.with_name(path.name + ".tmp")
    torch.save(state, temp)
    os.replace(temp, path)

def restore_resume(path, model, optimizer, scheduler, scaler):
    state = torch.load(path, map_location="cpu", weights_only=False)  # Own trusted checkpoint only.
    named = dict(model.named_parameters())
    with torch.no_grad():
        for n, value in state["trainable"].items():
            assert named[n].shape == value.shape
            named[n].copy_(value.to(named[n].device))
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    if state["allocator"] is not None:
        for k, values in state["allocator"].items():
            device = next(model.parameters()).device
            setattr(model.base_model.rankallocator, k, {n: t.to(device) for n, t in values.items()})
        model.peft_config["default"].rank_pattern = state["rank_pattern"]
    random.setstate(state["python_rng"])
    np.random.set_state(state["numpy_rng"])
    torch.set_rng_state(state["torch_rng"])
    if CUDA:
        torch.cuda.set_rng_state_all(state["cuda_rng"])
    return state

def optimizer_group_indices(step):
    effective = CFG["micro_batch"] * CFG["grad_accum"]
    per_epoch = len(ENCODED_TRAIN) // effective
    epoch, group = divmod(step, per_epoch)
    order = np.random.default_rng(CFG["seed"] + epoch).permutation(len(ENCODED_TRAIN))
    return order[group*effective:(group+1)*effective]

def train_adapter(method, rank, folder):
    effective = CFG["micro_batch"] * CFG["grad_accum"]
    total_steps = CFG["epochs"] * (len(ENCODED_TRAIN) // effective)
    assert total_steps == 450, "Default protocol expects 450 updates. Review schedule before changing settings."
    model = optimizer = scheduler = scaler = None
    try:
        seed_all(CFG["seed"])
        model = training_model(method, rank, total_steps)
        trainable, total = model.get_nb_trainable_parameters()  # PEFT counts logical 4-bit weight elements.
        param_stats = {"trainable_parameters": int(trainable), "total_parameters": int(total),
                       "trainable_pct": 100 * trainable / total}
        # Denominator is complete logical model incl. adapter, not packed NF4 storage elements.
        save_json(folder / "adapter_config_requested.json", model.peft_config["default"].to_dict())
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
            lr=CFG["lr"], betas=tuple(CFG["adam_betas"]), weight_decay=CFG["weight_decay"])
        scheduler = get_linear_schedule_with_warmup(optimizer,
            num_warmup_steps=math.ceil(CFG["warmup_ratio"]*total_steps), num_training_steps=total_steps)
        scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
        checkpoint = folder / "resume_state.pt"
        step, logs, previous_seconds, previous_peak = 0, [], 0.0, 0
        if checkpoint.exists():
            state = restore_resume(checkpoint, model, optimizer, scheduler, scaler)
            step, logs = state["next_step"], state["logs"]
            previous_seconds, previous_peak = state["training_seconds"], state["peak_bytes"]
            del state
            print(f"Resumed {method} rank {rank} at update {step}/{total_steps}.")
        assert 0 <= step <= total_steps
        model.train()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        overflow_retries = 0
        while step < total_steps:
            indices = optimizer_group_indices(step)
            group_loss = 0.0
            for offset in range(0, len(indices), CFG["micro_batch"]):
                records = [ENCODED_TRAIN[int(i)] for i in indices[offset:offset+CFG["micro_batch"]]]
                batch = collate_examples(records)
                with autocast_context():
                    loss = model(**batch).loss
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Nonfinite loss at update {step}; inspect precision/data.")
                group_loss += float(loss.detach()) / CFG["grad_accum"]
                scaler.scale(loss / CFG["grad_accum"]).backward()
                del batch, loss
            scaler.unscale_(optimizer)
            norm = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], CFG["max_grad_norm"])
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.get_scale() < old_scale
            if skipped:
                optimizer.zero_grad(set_to_none=True)
                overflow_retries += 1
                if overflow_retries > 20:
                    raise FloatingPointError("Repeated FP16 overflows; stop rather than silently changing precision.")
                print("FP16 overflow; retrying same optimizer group with reduced loss scale.")
                continue
            if not torch.isfinite(norm):
                raise FloatingPointError("Nonfinite gradient norm.")
            overflow_retries = 0
            step += 1
            if method == "AdaLoRA":
                model.base_model.update_and_allocate(step)
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            log = {"step": step, "epoch": step/(total_steps/CFG["epochs"]),
                   "loss": group_loss, "lr": scheduler.get_last_lr()[0]}
            if method == "AdaLoRA":
                pattern = model.peft_config["default"].rank_pattern
                log["active_rank_slots"] = sum(sum(v) for v in pattern.values()) if pattern else None
            logs.append(log)
            if step % 10 == 0 or step == total_steps:
                print(f"{method} r={rank}: step {step}/{total_steps}, loss={group_loss:.4f}")
            if CFG["save_resume_checkpoints"] and (step % CFG["checkpoint_every"] == 0 or step == total_steps):
                torch.cuda.synchronize()
                elapsed = previous_seconds + time.perf_counter() - start
                peak = max(previous_peak, torch.cuda.max_memory_allocated())
                save_resume(checkpoint, model, optimizer, scheduler, scaler, step, logs, elapsed, peak)
                save_jsonl(folder / "training_log.jsonl", logs)
        torch.cuda.synchronize()
        seconds = previous_seconds + time.perf_counter() - start
        peak = max(previous_peak, torch.cuda.max_memory_allocated())
        if method == "AdaLoRA":
            pattern = model.peft_config["default"].rank_pattern
            assert pattern, "AdaLoRA did not allocate ranks."
            save_json(folder / "final_rank_pattern.json", pattern)
            param_stats["active_rank_slots"] = int(sum(sum(v) for v in pattern.values()))
            param_stats["ranked_matrices"] = len(pattern)
        model.save_pretrained(folder / "adapter", safe_serialization=True)
        tokenizer.save_pretrained(folder / "adapter")
        save_jsonl(folder / "training_log.jsonl", logs)
        stats = {**param_stats, "total_optimizer_steps": total_steps, "training_minutes": seconds/60,
                 "peak_gpu_gib": peak/2**30, "training_loss": float(np.mean([x["loss"] for x in logs])),
                 "final_training_loss": logs[-1]["loss"],
                 "timing_policy": "synchronized optimizer loop incl periodic checkpoint I/O; excludes load/final adapter save/eval"}
        save_json(folder / "training_stats.json", stats)
        return stats
    finally:
        del model, optimizer, scheduler, scaler
        cleanup()

def run_experiment(method, rank=8):
    assert method in ("LoRA", "QLoRA", "DoRA", "AdaLoRA") and rank in (4, 8, 16, 32)
    assert completed("Original", 0), "Run the original baseline first; no training has started."
    if completed(method, rank):
        print("Already complete, reusing:", run_dir(method, rank))
        return read_json(run_dir(method, rank) / "metrics.json")
    folder = ensure_run(method, rank)
    model = None
    try:
        if (folder / "training_stats.json").exists():
            assert (folder / "adapter" / "adapter_model.safetensors").exists(), "Missing final adapter."
            stats = read_json(folder / "training_stats.json")
            print("Training complete; resuming evaluation only.")
        else:
            stats = train_adapter(method, rank, folder)
        cleanup()
        # Load exactly the saved checkpoint for both selection and final test evidence.
        model = load_base(method)
        model = PeftModel.from_pretrained(model, folder / "adapter", is_trainable=False)
        seed_all(CFG["seed"])
        selection = eval_model(model, SELECTION, folder / "selection_predictions.jsonl")
        test = eval_model(model, TEST, folder / "test_predictions.jsonl")
        result = {"status": "complete", "method": method, "rank": rank, **stats,
            "selection": selection, "test": test,
            "test_predictions_hash": digest(read_jsonl(folder / "test_predictions.jsonl")),
            "selection_predictions_hash": digest(read_jsonl(folder / "selection_predictions.jsonl")),
            "identity_hash": digest(run_identity(method, rank)), "adapter_path": str(folder / "adapter"),
            "adapter_file_hashes": {name: file_digest(folder / "adapter" / name)
                for name in ("adapter_model.safetensors", "adapter_config.json")}}
        save_json(folder / "metrics.json", result)  # Only complete after ALL 200 predictions.
        print(json.dumps(result, indent=2))
        return result
    except torch.cuda.OutOfMemoryError as exc:
        save_json(folder / "error.json", {"error": "CUDA OOM", "method": method, "rank": rank})
        raise RuntimeError(f"{method} r={rank} OOM. No precision/model substitution. Use larger GPU; "
                           "or start a NEW output directory with a smaller shared sequence budget.") from exc
    except Exception as exc:
        save_json(folder / "error.json", {"error": type(exc).__name__, "detail": str(exc)})
        raise
    finally:
        del model
        cleanup()


## 9. Experiment 1 — LoRA

Frozen BF16/FP16 base; rank-8 low-rank direction updates.

Run setup/data and **both** Section 8 definition cells first. Enable only this control to run/resume independently.


In [ ]:
RUN_LORA = True
if RUN_LORA:
    run_experiment("LoRA", rank=8)
else:
    print("LoRA disabled; enable this cell to run/resume.")


## 10. Experiment 2 — QLoRA

Actual 4-bit NF4 base with double quantization, hardware-aware compute dtype, k-bit preparation and rank-8 adapters. NF4 assertions fail if quantization is absent.

Run setup/data and **both** Section 8 definition cells first. Enable only this control to run/resume independently.


In [ ]:
RUN_QLORA = True
if RUN_QLORA:
    run_experiment("QLoRA", rank=8)
else:
    print("QLoRA disabled; enable this cell to run/resume.")


## 11. Experiment 3 — DoRA

PEFT `use_dora=True` with rank 8; verifies magnitude-vector parameters exist. No substitution with ordinary LoRA.

Run setup/data and **both** Section 8 definition cells first. Enable only this control to run/resume independently.


In [ ]:
RUN_DORA = True
if RUN_DORA:
    run_experiment("DoRA", rank=8)
else:
    print("DoRA disabled; enable this cell to run/resume.")


## 12. Experiment 4 — AdaLoRA

PEFT AdaLoraConfig with initial rank 12 and target average budget 8. Every successful optimizer step updates importance and allocation before gradients are cleared. Trainable counts report allocated initial tensors; the final active rank pattern is exported separately.

Run setup/data and **both** Section 8 definition cells first. Enable only this control to run/resume independently.


In [ ]:
RUN_ADALORA = True
if RUN_ADALORA:
    run_experiment("AdaLoRA", rank=8)
else:
    print("AdaLoRA disabled; enable this cell to run/resume.")


## 13. Five-row results comparison

These cells reload persisted evidence. Missing runs are shown with a status and blank numerical
fields, not fabricated zeros. CSV/plots for the final five-way comparison require all five runs.
Inference speed includes the QLoRA NF4 representation and unmerged adapters, making deployment
representation explicit. Baseline trainable count/percentage, training memory/time/loss are N/A.


In [ ]:
METHODS = ["LoRA", "QLoRA", "DoRA", "AdaLoRA"]

def result_row(result):
    test = result["test"]
    return {"Method": result["method"], "Adapter rank": result["rank"],
        "Trainable parameters": result["trainable_parameters"], "Trainable percentage (%)": result["trainable_pct"],
        "Peak GPU memory (GiB)": result["peak_gpu_gib"], "Training duration (min)": result["training_minutes"],
        "Training loss": result["training_loss"], "JSON validity (%)": test["json_pct"],
        "Tool accuracy (%)": test["tool_pct"], "Full match accuracy (%)": test["exact_pct"],
        "Inference speed (tokens/s)": test["tokens_per_second"],
        "Inference speed (requests/s)": test["requests_per_second"],
        "Test examples": test["n"], "Status": result["status"]}

def load_required_results():
    specs = [("Original", 0)] + [(m, 8) for m in METHODS]
    missing = [m for m, r in specs if not completed(m, r)]
    if missing:
        print("Unfinished runs:", ", ".join(missing))
        return None
    return [read_json(run_dir(m, r) / "metrics.json") for m, r in specs]

def comparison_table():
    results = load_required_results()
    if results is None:
        return None
    frame = pd.DataFrame([result_row(x) for x in results])
    assert len(frame) == 5 and (frame["Test examples"] == 200).all()
    frame.to_csv(OUTPUT_DIR / "tables" / "comparison.csv", index=False, na_rep="N/A")
    display(frame.fillna("N/A"))
    return frame

comparison = comparison_table()


## 14. Best-method selection

**Rank-study winner:** select using only the predeclared 180-example training-side generation
scores. This subset was trained on, so selection scores are optimistic; they are never presented
as validation/generalization accuracy. Reusing the test set for method/rank tuning is prohibited.
Ties: tool accuracy, full match, JSON validity descending, then trainable parameters and method
name ascending. Rank 8 is reused rather than retrained. All four base experiments must finish.


In [ ]:
def choose_rank_method():
    results = load_required_results()
    if results is None:
        return None
    candidates = [x for x in results if x["method"] != "Original"]
    for result in candidates:
        rows = read_jsonl(run_dir(result["method"], 8) / "selection_predictions.jsonl")
        assert [x["id"] for x in rows] == SELECTION_IDS
        assert digest(rows) == result["selection_predictions_hash"]
    best = sorted(candidates, key=lambda x: (-x["selection"]["tool_pct"],
        -x["selection"]["exact_pct"], -x["selection"]["json_pct"],
        x["trainable_parameters"], x["method"]))[0]
    decision = {"method": best["method"], "rank": 8, "rule": CFG["selection_rule"],
                "selection": best["selection"], "selection_ids_hash": digest(SELECTION_IDS)}
    path = OUTPUT_DIR / "rank_method_selection.json"
    if path.exists():
        assert read_json(path) == decision, "Selection changed; inspect evidence and protocol."
    save_json(path, decision)
    print("Rank-study method:", best["method"], "(training-side selection only)")
    return best["method"]

rank_method = choose_rank_method()


## 15. Rank 4, 16 and 32 experiments

All three controls are enabled in this delivery copy. Run all executes them sequentially. Settings, examples,
seed, optimizer and generation remain fixed. Alpha scales with rank to preserve the alpha/r
ratio; keeping alpha constant would introduce a scaling change. AdaLoRA initial ranks are
`ceil(1.5×target)` = **6, 12, 24, 48**, with the same schedule fractions and total updates.
Thus the plotted AdaLoRA x-axis is target average rank; allocation can differ by matrix.


In [ ]:
RUN_RANK_4 = True
if RUN_RANK_4:
    method = choose_rank_method()
    if method is None:
        raise RuntimeError("Complete baseline and all four rank-8 experiments before the rank study.")
    run_experiment(method, rank=4)
else:
    print("Additional rank 4 disabled; enable this cell to run/resume.")


In [ ]:
RUN_RANK_16 = True
if RUN_RANK_16:
    method = choose_rank_method()
    if method is None:
        raise RuntimeError("Complete baseline and all four rank-8 experiments before the rank study.")
    run_experiment(method, rank=16)
else:
    print("Additional rank 16 disabled; enable this cell to run/resume.")


In [ ]:
RUN_RANK_32 = True
if RUN_RANK_32:
    method = choose_rank_method()
    if method is None:
        raise RuntimeError("Complete baseline and all four rank-8 experiments before the rank study.")
    run_experiment(method, rank=32)
else:
    print("Additional rank 32 disabled; enable this cell to run/resume.")


## 16. Rank comparison and plot

The graph reports held-out test tool accuracy as a descriptive rank comparison. Do not use it
to change settings and then claim an untouched test estimate. One seed and 200 examples limit
precision: one correct prediction changes accuracy by 0.5 percentage points. The generated
summary describes observed changes only; it does not infer statistical significance.


In [ ]:
def rank_table_and_plot():
    method = choose_rank_method()
    if method is None:
        return None
    missing = [r for r in (4, 8, 16, 32) if not completed(method, r)]
    if missing:
        print("Missing rank runs:", missing)
        return None
    results = [read_json(run_dir(method, r) / "metrics.json") for r in (4, 8, 16, 32)]
    table = pd.DataFrame([result_row(x) for x in results]).sort_values("Adapter rank")
    table.to_csv(OUTPUT_DIR / "tables" / "rank_comparison.csv", index=False)
    display(table)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(table["Adapter rank"], table["Tool accuracy (%)"], "o-", label=method)
    ax.set(xticks=[4, 8, 16, 32], ylim=(0, 100), xlabel="Adapter rank (AdaLoRA: target average)",
           ylabel="Tool selection accuracy (%)", title=f"{method}: tool accuracy versus adapter rank")
    ax.grid(alpha=.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "graphs" / "tool_accuracy_vs_rank.png", dpi=160)
    plt.show()
    plt.close(fig)
    lines = [f"Method: {method}. Test size: 200. One seed; no significance claim."]
    by_rank = table.set_index("Adapter rank")
    for low, high in ((4, 8), (8, 16), (16, 32)):
        delta = by_rank.loc[high, "Tool accuracy (%)"] - by_rank.loc[low, "Tool accuracy (%)"]
        memory = by_rank.loc[high, "Peak GPU memory (GiB)"] - by_rank.loc[low, "Peak GPU memory (GiB)"]
        lines.append(f"Rank {low}→{high}: tool accuracy {delta:+.2f} percentage points; "
                     f"peak allocated memory {memory:+.3f} GiB.")
    summary = "\n".join(lines)
    (OUTPUT_DIR / "rank_summary.txt").write_text(summary, encoding="utf-8")
    print(summary)
    return table

rank_comparison = rank_table_and_plot()


## 17. Error analysis

For reporting only, choose the best **test** model from the original baseline, four adapted
rank-8 runs and three additional rank runs by tool accuracy, then full match, JSON validity,
fewer trainable parameters (the frozen original has none), then method name/rank.
This is explicitly a post-hoc descriptive label, not further tuning.
Extract the first five non-exact predictions in the fixed test order; fewer errors produce fewer
rows. Categories are structural heuristics, not a claim of semantic authorization judgment.
Any changed/missing/extra argument counts as an error. Review the saved evidence manually.


In [ ]:
def error_category(row):
    parsed = row["parsed"]
    if not row["json_valid"]:
        return "Invalid/broken JSON"
    if not row["call_shape_valid"]:
        return "Missing tool call or invalid call structure"
    if parsed["name"] != row["expected"]["name"]:
        if any(word in parsed["name"].lower() for word in ("refund", "delete", "cancel", "purchase")):
            return "Wrong tool; potentially unnecessary action (manual review)"
        return "Wrong tool"
    expected_args, predicted_args = row["expected"]["arguments"], parsed["arguments"]
    if set(predicted_args) - set(expected_args):
        return "Hallucinated argument"
    if set(expected_args) - set(predicted_args):
        return "Missing argument"
    return "Incorrect argument value or type"

def export_error_analysis():
    method = choose_rank_method()
    if method is None or any(not completed(method, r) for r in (4, 8, 16, 32)):
        print("Finish the rank study before final best-model error analysis.")
        return None
    specs = [("Original", 0)] + [(m, 8) for m in METHODS] + [(method, r) for r in (4, 16, 32)]
    results = [read_json(run_dir(m, r) / "metrics.json") for m, r in specs]
    best = sorted(results, key=lambda x: (-x["test"]["tool_pct"], -x["test"]["exact_pct"],
        -x["test"]["json_pct"], x["trainable_parameters"] or 0, x["method"], x["rank"] or 0))[0]
    predictions = read_jsonl(run_dir(best["method"], best["rank"]) / "test_predictions.jsonl")
    failures = [x for x in predictions if not x["exact_match"]]
    rows = [{"Example ID": x["id"], "Customer/user message": x["user"],
        "Correct function call": canonical(x["expected"]), "Actual generated response": x["raw_response"],
        "Expected arguments": canonical(x["expected"]["arguments"]),
        "Predicted arguments": canonical(x["parsed"]["arguments"]) if x["call_shape_valid"] else "unavailable",
        "Error category": error_category(x), "Method": best["method"], "Rank": best["rank"]}
        for x in failures[:5]]
    columns = ["Example ID", "Customer/user message", "Correct function call", "Actual generated response",
               "Expected arguments", "Predicted arguments", "Error category", "Method", "Rank"]
    frame = pd.DataFrame(rows, columns=columns)
    frame.to_csv(OUTPUT_DIR / "tables" / "error_analysis.csv", index=False)
    summary = {"method": best["method"], "rank": best["rank"], "total_test_errors": len(failures),
        "shown_errors": len(rows), "all_error_categories": dict(Counter(error_category(x) for x in failures)),
        "selection_caveat": "Best test row is post-hoc, descriptive; no subsequent fitting."}
    save_json(OUTPUT_DIR / "error_analysis_summary.json", summary)
    print(f"Best descriptive test row: {best['method']} rank {best['rank']}. "
          f"Showing {len(rows)} of {len(failures)} actual non-exact outputs.")
    display(frame)
    print("All error categories:", summary["all_error_categories"])
    return frame

failure_analysis = export_error_analysis()


## 18. Final graphs and exported results

Generate only from complete saved metrics. Baseline has no training memory/time bar because
no training occurred. Peak memory means PyTorch allocated memory, not total device reservation
or nvidia-smi utilization; AdaLoRA's temporary importance/orthogonality buffers are included.


In [ ]:
def export_final_graphs():
    table = comparison_table()
    if table is None:
        return
    specs = [
        ("JSON validity (%)", "JSON validity on 200 test examples", "json_validity_comparison.png", False),
        ("Tool accuracy (%)", "Tool selection accuracy on 200 test examples", "tool_accuracy_comparison.png", False),
        ("Full match accuracy (%)", "Exact function and argument match on 200 test examples", "full_match_comparison.png", False),
        ("Peak GPU memory (GiB)", "Peak GPU memory during adapter training", "peak_gpu_memory_comparison.png", True),
        ("Training duration (min)", "Measured adapter training duration", "training_duration_comparison.png", True),
    ]
    for column, title, filename, training_only in specs:
        frame = table[table["Method"] != "Original"] if training_only else table
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(frame["Method"], frame[column], color=["#334155", "#0284c7", "#0d9488", "#d97706", "#9333ea"][:len(frame)])
        ax.set(title=title, xlabel="Method (adapters unmerged)", ylabel=column)
        if "(%)" in column:
            ax.set_ylim(0, 100)
        ax.grid(axis="y", alpha=.2)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "graphs" / filename, dpi=160)
        plt.show()
        plt.close(fig)
    rank_table_and_plot()

export_final_graphs()


## 19. Experiment summary and assignment-report evidence

The export below assembles measured evidence and report prompts; it does not invent a
production recommendation or surprises. Write the final report (at most four pages) using
the real trade-offs. Address domain mismatch, the in-sample selection proxy, 200-test size,
single seed, and any differences in GPU/run precision. Runs on a changed GPU are rejected by
identity checks; use a separate study directory rather than combine unmatched speed figures.

**Deployment evidence boundaries:** measured latency is unmerged. PEFT supports merging
DoRA magnitude/direction adaptations into dense base weights, avoiding adapter forward
overhead. NF4 QLoRA merging can introduce rounding differences; dequantizing/merging into
dense weights loses the NF4 storage advantage, and requantization requires fresh validation.
Do not claim a measured post-merge speedup: this notebook has not benchmarked merged models.
See [PEFT LoRA/DoRA guide](https://huggingface.co/docs/peft/v0.17.0/en/developer_guides/lora)
and [4-bit quantization guide](https://huggingface.co/docs/transformers/v4.51.3/quantization/bitsandbytes).

**Agentic discussion:** one frozen base plus router/refund/shipping adapters shares the large
base allocation and stores only each small adapter, rather than three full model copies.
Serving still needs concurrency management; changing a global active adapter mid-request
can route a refund request through the wrong behavior. Bind adapter identity/version to each
request and synchronize swaps. Prediction alone does not authorize executing a refund.

**Sources:** [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct),
[Glaive data](https://huggingface.co/datasets/glaiveai/glaive-function-calling-v2),
[AdaLoRA API](https://huggingface.co/docs/peft/v0.17.0/package_reference/adalora),
[pinned allocator implementation](https://github.com/huggingface/peft/blob/v0.17.1/src/peft/tuners/adalora/model.py).
No actual GPU training was performed when this notebook was authored; authoring validation
is documented in the project README. Kaggle execution supplies the experiment results.


In [ ]:
def export_summary():
    base_results = load_required_results()
    method = choose_rank_method() if base_results is not None else None
    if method is None or any(not completed(method, r) for r in (4, 8, 16, 32)):
        print("Summary not finalized: complete all five base rows and the three additional rank runs.")
        return
    comparison_table()
    rank_table_and_plot()
    export_error_analysis()
    baseline = base_results[0]["test"]
    lines = ["Assignment Beta Question 2 — measured evidence", "",
        f"Model: {CFG['model_id']} @ {PROTOCOL['model_sha']}",
        f"Dataset: {CFG['dataset_id']} @ {PROTOCOL['dataset_sha']}",
        f"Scope: {MANIFEST['scope']}", MANIFEST["limitation"],
        f"Train/test: 1800/200; seed={CFG['seed']}; epochs={CFG['epochs']}; effective batch=8.",
        f"GPU: {HARDWARE['gpu']}; compute dtype: {HARDWARE['compute_dtype']}.",
        "", "Measured changes versus the original model:"]
    for result in base_results[1:]:
        test = result["test"]
        lines.append(f"{result['method']}: tool accuracy {test['tool_pct']:.2f}% "
            f"({test['tool_pct']-baseline['tool_pct']:+.2f} pp); full match {test['exact_pct']:.2f}% "
            f"({test['exact_pct']-baseline['exact_pct']:+.2f} pp); "
            f"training memory {result['peak_gpu_gib']:.3f} GiB; "
            f"training {result['training_minutes']:.2f} min; "
            f"unmerged inference {test['tokens_per_second']:.2f} generated tokens/s.")
    lines += ["", f"Training-side selected rank-study method: {method}.",
        (OUTPUT_DIR / "rank_summary.txt").read_text(encoding="utf-8"), "",
        "Report prompts (write your own interpretation):",
        "1. Production choice: weigh full match, unnecessary actions, memory, time and unmerged speed.",
        "2. Use error_analysis.csv and all_error_categories; review five actual mistakes.",
        "3. Explain shared-base multi-agent adapters and live adapter-swap race/version risk.",
        "4. State a specific observed surprise using numerical differences above.",
        "5. Compare dense DoRA merge and NF4 QLoRA merge/requantization; no merged speed was measured.",
        "Limitations: generated general benchmark, one seed, small test, in-sample selection, post-hoc best row.",
        "Do not claim authorized-action safety or customer-support deployment readiness from exact-match alone."]
    summary = "\n".join(lines)
    (OUTPUT_DIR / "experiment_summary.txt").write_text(summary, encoding="utf-8")
    save_json(OUTPUT_DIR / "export_index.json", {
        "protocol_hash": digest(PROTOCOL), "dataset_manifest": "dataset/manifest.json",
        "comparison": "tables/comparison.csv", "rank_comparison": "tables/rank_comparison.csv",
        "error_analysis": "tables/error_analysis.csv", "summary": "experiment_summary.txt",
        "rank_method": method, "all_base_results": base_results,
        "rank_runs": [read_json(run_dir(method, r)/"metrics.json") for r in (4, 8, 16, 32)]})
    print(summary)
    print("All evidence exported under:", OUTPUT_DIR)

export_summary()


## Automatic four-page measured report export
This final step requires all experiments and their exported tables. The PDF is generated from real evidence. It fails if experiments are incomplete.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","reportlab>=4.4,<5"])

In [ ]:
"""Four-page Q2 report, populated only with actual saved GPU evidence."""
from pathlib import Path
import csv,json
from xml.sax.saxutils import escape

def build_q2_report(study_dir, output_pdf, require_complete=False):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet,ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate,Paragraph,Spacer,Table,TableStyle,PageBreak
    root=Path(study_dir);output=Path(output_pdf);output.parent.mkdir(parents=True,exist_ok=True)
    idx=root/'export_index.json'; complete=idx.exists()
    if require_complete and not complete:raise RuntimeError('Complete all GPU experiments before exporting the measured report.')
    evidence=json.loads(idx.read_text()) if complete else None
    styles=getSampleStyleSheet()
    styles.add(ParagraphStyle(name='BodyQ',fontName='Helvetica',fontSize=9.4,leading=13.1,spaceAfter=8))
    styles.add(ParagraphStyle(name='SmallQ',fontName='Helvetica',fontSize=7.6,leading=10,spaceAfter=5))
    story=[]
    def p(s,style='BodyQ'):story.append(Paragraph(s,styles[style]))
    def h(s):p(s,'Heading2')
    def table(rows,widths=None):
        cells=[[Paragraph(escape(str(c)),styles['SmallQ']) for c in row] for row in rows]
        t=Table(cells,colWidths=widths,repeatRows=1,hAlign='LEFT')
        t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#e6eef5')),('VALIGN',(0,0),(-1,-1),'TOP'),('LINEBELOW',(0,0),(-1,0),.6,colors.HexColor('#16334d')),('BOTTOMPADDING',(0,0),(-1,-1),6),('TOPPADDING',(0,0),(-1,-1),5)]));story.append(t)
    p('Question 2 | PEFT deployment report','Title')
    p('Muhammad Sabeeh | 23K-0002 | Qwen2.5-1.5B-Instruct | 1,800 train / 200 test','SmallQ')
    if not complete:p('<b>Evidence status: GPU experiments have not run. This is an evidence-limited report, not a completed empirical comparison.</b> No experimental score, duration, inference speed or GPU memory is inferred from a tiny CPU test.')
    h('1. Production choice and deployment trade-offs')
    if complete:
        runs=evidence['all_base_results'][1:]
        winner=sorted(runs,key=lambda r:(-r['test']['exact_pct'],-r['test']['tool_pct'],-r['test']['json_pct'],r['peak_gpu_gib'],r['training_minutes']))[0]
        p(f"The strongest measured base comparison is <b>{escape(winner['method'])}, rank {winner['rank']}</b>: full match {winner['test']['exact_pct']:.2f}%, tool accuracy {winner['test']['tool_pct']:.2f}%, JSON validity {winner['test']['json_pct']:.2f}%, peak training memory {winner['peak_gpu_gib']:.3f} GiB, training {winner['training_minutes']:.2f} minutes, unmerged generation {winner['test']['tokens_per_second']:.2f} tokens/s. This post-hoc recommendation uses the reported test evidence; it is not additional training or a deployment safety certification.")
    else:
        p('My provisional engineering choice is rank-8 LoRA on a dense BF16/FP16 base, provided its measured full match is competitive and it fits the production GPU. Its small adapter and simple dense merge are attractive for a small team. This is a conditional design preference; the study does not yet identify an empirical winner. If dense storage is the binding constraint, NF4 QLoRA is the first alternative to evaluate.')
    p('Use full match and unnecessary-action errors ahead of JSON validity alone. A perfectly valid refund call can still be the wrong action. Compare training memory and duration separately from serving throughput: a training-memory advantage does not imply faster inference. Keep tool names and argument schemas constrained, validate order ownership and refund policy in backend code, and require confirmation for consequential actions.')
    p('A dense LoRA adapter can be merged into its dense base, removing the extra low-rank computation. Dense DoRA additionally learns magnitudes and can also be merged by supported PEFT implementations. NF4 QLoRA starts from quantized weights: merging may require dequantization or a supported quantized merge, and deployment may requantize. Rounding can change outputs and peak memory. No post-merge speed or quality gain is claimed without remeasurement.')
    h('Five-row comparison')
    rows=[['Method','Trainable / %','Peak GiB / min','JSON / tool / exact %']]
    if complete:
        for r in evidence['all_base_results']:
            fmt=lambda v:'N/A' if v is None else f'{v:.3f}'
            rows.append([r['method'],f"{r['trainable_parameters']} / {fmt(r['trainable_pct'])}",f"{fmt(r['peak_gpu_gib'])} / {fmt(r['training_minutes'])}",f"{r['test']['json_pct']:.1f} / {r['test']['tool_pct']:.1f} / {r['test']['exact_pct']:.1f}"])
    else:
        rows += [[m,'Unmeasured','N/A' if m=='Original' else 'Unmeasured','Unmeasured'] for m in ['Original','LoRA','QLoRA','DoRA','AdaLoRA']]
    table(rows,[75,119,100,188])
    story.append(PageBreak());h('2. Where it still breaks: five real errors')
    if complete:
        file=root/'tables'/'error_analysis.csv'; errors=list(csv.DictReader(file.open(encoding='utf-8')))
        summary=json.loads((root/'error_analysis_summary.json').read_text())
        p(f"Error-analysis row: {escape(str(summary['method']))}, rank {summary['rank']}. There are {summary['total_test_errors']} non-exact predictions in the 200 test records. This is a descriptive best-test-row selection, so its apparent performance is optimistic.")
        if not errors:p('The evaluated model made no full-match mistakes; five incorrect examples therefore do not exist in this test set.')
        for i,row in enumerate(errors[:5],1):
            p(f"<b>{i}. {escape(row['Error category'])}</b>",'SmallQ')
            for label,key in [('Message','Customer/user message'),('Correct','Correct function call'),('Actual','Actual generated response')]:
                p(f'<b>{label}:</b> {escape(row[key])}','SmallQ')
        p('Observed category counts: '+escape(json.dumps(summary['all_error_categories'])), 'SmallQ')
    else:
        p('Five real incorrect outputs cannot be selected until model generation runs. Fabricating customer messages and model responses would turn an error analysis into a simulation. The notebook automatically exports the first five non-exact responses from the descriptively best test row, including the full user message, expected call, raw response and error label.')
        table([['Evidence field','Collection rule'],['Customer message','Original held-out user message; preserve its wording.'],['Correct call','Frozen test JSON, including every argument and value type.'],['Actual response','Raw generated text; do not repair JSON before grading.'],['Error label','Broken JSON; invalid call structure; wrong tool; extra/missing argument; wrong value/type.'],['Pattern','Count all error categories across the entire 200-record set.']], [125,357])
        p('No pattern of actual model failures is claimed yet. The failure taxonomy specifically distinguishes syntactic reliability from semantic correctness: extra prose can break JSON, a hallucinated name can break dispatch, and a valid but unnecessary refund can break business policy. The last error requires policy-aware checks beyond exact match.')
    h('Protocol and leakage control')
    p('Freeze exactly 2,000 eligible unique Glaive examples with source/model revisions, then split 1,800/200 by seed. Duplicate normalized user messages cannot cross the split. Fit all 1,800 records with the same epochs, LR schedule, target modules, seed and masking. Use the fixed 180-example training-side subset to choose the rank-study method; it is in-sample and optimistic, not held-out validation. Never use the 200 test records to change training.')
    p('The exact four business tool names are not assumed to occur in Glaive. The notebook audits coverage and uses a general function-calling benchmark when the requested domain cannot supply 2,000 examples. It never relabels unrelated tools as support tools. Accordingly, this study cannot establish real refund authorization or Karachi-language performance.')
    story.append(PageBreak());h('3. The agentic angle: one base, several adapters')
    p('A router agent, refund agent and shipping agent can share one frozen base and load separate behavior adapters. The base is stored once, while each role stores only its small trainable update. Training also avoids gradients and optimizer state for the base. This cuts storage and optimizer memory relative to three fully fine-tuned copies, although activations, request KV caches and adapter-routing infrastructure still consume memory.')
    p('For one 4,096 by 4,096 matrix at rank 16, an adapter contains 131,072 weights versus 16,777,216 base weights. At FP16 this is 262,144 bytes (0.25 MiB), while the matrix alone occupies 33,554,432 bytes (32 MiB). Three base copies with one adapter each occupy 96.75 MiB; a shared base with three adapters occupies 32.75 MiB. This example excludes optimizer state and is not a whole-model memory benchmark.')
    p('A real live-swapping risk is request contamination: a shipping request may continue after the global active adapter is switched to refunds. Bind adapter identity and version to each request and decode operation; use a serving stack that isolates per-request adapters, or serialize access while switching. Cache keys, traces and rollback metadata must include adapter identity. Never merge one role permanently into the base and then treat the resulting weights as the common base for other roles.')
    h('Rank ablation: 4, 8, 16 and 32')
    if complete:
        rows=[['Rank','Tool %','Exact %','Peak GiB','Minutes']]
        for r in evidence['rank_runs']:rows.append([r['rank'],f"{r['test']['tool_pct']:.2f}",f"{r['test']['exact_pct']:.2f}",f"{r['peak_gpu_gib']:.3f}",f"{r['training_minutes']:.2f}"])
        table(rows,[55,95,95,105,132]);p(escape((root/'rank_summary.txt').read_text(encoding='utf-8')))
    else:
        p('The rank grid is fixed at 4, 8, 16 and 32. Rank 8 reuses the base comparison adapter, so the full experiment needs seven training runs plus the original-model evaluation. AdaLoRA uses an initial-rank policy ceil(1.5 times target), including 12 to target 8. Each run uses the same data and shared schedule. Plot the actual 200-record test tool accuracy against rank only after completion.')
        p('I cannot conclude that doubling rank helps without that graph. The required conclusion will compare percentage-point gains against additional trainable parameters, peak memory and training time. With only 200 test records, one changed decision is 0.5 percentage points; small changes may reflect sampling noise rather than a meaningful production advantage.')
    h('Measurement discipline')
    p('Use torch.cuda.max_memory_allocated() after resetting peak counters for training. CUDA synchronization brackets measured work. Record real elapsed training minutes, and score all 200 raw generations with strict whole-output JSON parsing and exact argument matching. Generation throughput is batch-one, greedy and unmerged; model loading and tokenization are excluded. These choices must be identical across methods.')
    story.append(PageBreak());h('4. What surprised me, and what follows')
    if complete:
        base=evidence['all_base_results'];orig=base[0];changes=sorted(base[1:],key=lambda r:abs(r['test']['tool_pct']-orig['test']['tool_pct']),reverse=True)
        r=changes[0];p(f"The largest observed change from the base model was {escape(r['method'])}: {r['test']['tool_pct']-orig['test']['tool_pct']:+.2f} percentage points in tool accuracy, alongside {r['test']['exact_pct']-orig['test']['exact_pct']:+.2f} points in full match. The difference between these two gains shows why tool-name accuracy alone cannot summarize argument correctness. This is the measured observation; no invented prior expectation is attributed to the student.")
    else:
        p('A concrete preparation finding was that the existing full-corpus validation found 112,960 Glaive conversations, 5,164 eligible unique examples under the fixed parser/token rules, and zero eligible examples for the exact four requested business tools. This is a data-preparation finding, not a fine-tuning result. It changes the interpretation of the experiment: a generic function-calling gain is not automatically a customer-support gain.')
        p('There is no training-result surprise to report yet. In particular, it would be unjustified to say that AdaLoRA beat LoRA, that QLoRA saved a measured amount of VRAM, or that rank 32 was wasteful. The notebook appends a measured report after the actual runs; this report keeps the missing evidence visible.')
    h('Shipping decision')
    p('For a real release, retain backend tool authorization, schema validation, user confirmation for transactions, and a human fallback even if an adapter scores highly. Evaluate a separate support set containing ambiguous requests, missing order IDs, unauthorized refund requests, Roman Urdu, Urdu script and English. The generated Glaive benchmark supplies a reproducible experiment, but it does not replace a representative deployment evaluation.')
    h('Limitations and completion gate')
    p('One seed, a small test set, train-side method selection and post-hoc best-test-row reporting limit inference. Core library versions and GPU identity are checked before resuming. The four methods share the optimizer and schedule; QLoRA differs in the required base quantization and AdaLoRA adds its intrinsic rank schedule/orthogonality behavior. No post-merge serving benchmark is included.')
    if not complete:p('<b>Remaining empirical work:</b> run this notebook on a CUDA GPU, preserve all eight evaluated rows, export comparison/rank graphs and five genuine errors, then regenerate this report. The current machine has no CUDA device. The program deliberately refuses to manufacture the missing measurements.')
    h('References and AI declaration')
    for text in ['Hu et al., LoRA (2021), https://arxiv.org/abs/2106.09685',
                 'Dettmers et al., QLoRA (2023), https://arxiv.org/abs/2305.14314',
                 'Liu et al., DoRA (2024), https://arxiv.org/abs/2402.09353',
                 'Zhang et al., AdaLoRA (2023), https://arxiv.org/abs/2303.10512',
                 'Hugging Face PEFT documentation, https://huggingface.co/docs/peft/']:
        p(escape(text),'SmallQ')
    p('OpenAI Codex assisted with implementation, arithmetic, research and report writing. No AI-produced experimental score is presented as an observed result.','SmallQ')
    def footer(c,doc):
        c.setFont('Helvetica',8);c.setFillColor(colors.HexColor('#526779'));c.drawString(48,28,'Assignment Beta | Q2 | Muhammad Sabeeh | 23K-0002');c.drawRightString(A4[0]-48,28,str(doc.page))
    SimpleDocTemplate(str(output),pagesize=A4,rightMargin=48,leftMargin=48,topMargin=38,bottomMargin=44).build(story,onFirstPage=footer,onLaterPages=footer)
    return str(output)

if False:
    import argparse
    p=argparse.ArgumentParser();p.add_argument('study_dir');p.add_argument('output_pdf');p.add_argument('--require-complete',action='store_true');a=p.parse_args()
    print(build_q2_report(a.study_dir,a.output_pdf,a.require_complete))

build_q2_report(OUTPUT_DIR, OUTPUT_DIR / 'Question_2_Report.pdf', require_complete=True)


## Download the complete Kaggle results
Run this after the measured PDF export finishes. Download the ZIP and the executed notebook before ending your session.


In [ ]:
import shutil
from IPython.display import FileLink, display

required = [OUTPUT_DIR / "export_index.json", OUTPUT_DIR / "Question_2_Report.pdf"]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise RuntimeError("Finish all experiments and the measured report export first. Missing: " + ", ".join(missing))

archive_path = shutil.make_archive(
    "/kaggle/working/Assignment_Beta_Q2_results", "zip", root_dir=str(OUTPUT_DIR)
)
print("Results ZIP:", archive_path)
print("Also download this notebook with its executed outputs.")
display(FileLink(archive_path))
